In [59]:
import xml.etree.ElementTree as ET
import re
import html
import json
from bs4 import BeautifulSoup
import random
from collections import defaultdict

In [ ]:
# Parse the full XML to list of python dicts
#will serve as data sturcture for rest of the project

In [2]:
file_path = 'mplus_topics_2026-07-07.xml'

In [3]:
tree = ET.parse(file_path)
root = tree.getroot()
topics = root.findall('health-topic')

In [4]:
print("topics found:", len(topics))
print("expected total (from root attrib):", root.attrib.get('total'))

topics found: 2033
expected total (from root attrib): 2033


Just reconfirming the parse is complete

In [17]:
all_topics = []

for topic in topics: 
#groups and also-called can appear multiple times per topic, so these need to stay as lists, not single values
    groups = [g.text for g in topic.findall('group') if g.text and g.text.strip()]
    also_called = [a.text for a in topic.findall('also-called') if a.text and a.text.strip()]

    # site elements - url + title might be useful later to sight original source
    sites = []
    for site in topic.findall('site'):
        title = site.get('title')
        #try attribute first then fallback to text 
        url = site.get('url') or site.text or ""
        if url:
            url = url.strip()
        sites.append({
            'title': title,
            'url': url
        })
    
    summary_el = topic.find('full-summary')
    summary_raw = summary_el.text if (summary_el is not None and summary_el.text) else ""

    record = {
        'id': topic.get('id'),
        'title': topic.get('title'),
        'url': topic.get('url'),
        'language': topic.get('language'),
        'meta_desc': topic.get('meta-desc'),
        'also_called': also_called,
        'groups': groups,
        'sites': sites,
        'summary_raw': summary_raw.strip() if summary_raw else "",  # keeping this untouched for now
    }
    all_topics.append(record)

In [18]:
print("parsed", len(all_topics))

parsed 2033


In [19]:
all_topics


[{'id': '6308',
  'title': 'A1C',
  'url': 'https://medlineplus.gov/a1c.html',
  'language': 'English',
  'meta_desc': 'If you are being tested for Type 2 diabetes, your doctor gives you an A1C test. The test is also used to monitor your A1C levels.',
  'also_called': ['Glycohemoglobin', 'HbA1C', 'Hemoglobin A1C test'],
  'groups': ['Diagnostic Tests', 'Diabetes Mellitus'],
  'sites': [{'title': 'A1C and eAG',
    'url': 'https://diabetes.org/a1c-eag-conversion-calculator'},
   {'title': 'A1C test',
    'url': 'https://medlineplus.gov/ency/article/003640.htm'},
   {'title': 'A1C Test and Diabetes',
    'url': 'https://www.niddk.nih.gov/health-information/diagnostic-tests/a1c-test'},
   {'title': 'A1C Test and Race/Ethnicity',
    'url': 'https://www.niddk.nih.gov/health-information/diagnostic-tests/a1c-test-race-ethnicity'},
   {'title': 'ClinicalTrials.gov: Hemoglobin A1C',
    'url': 'https://clinicaltrials.gov/search?intr=%22Hemoglobin+A1C%22&aggFilters=status:not%20rec'},
   {'titl

All topics have been parsed into the python dictionaries.

In [24]:
def clean_summary(raw_text):
    if not raw_text or not isinstance(raw_text, str):
        return ""

    #  Use BeautifulSoup - much more reliable for HTML
    soup = BeautifulSoup(raw_text, "html.parser")
    
    # Extract text while preserving some structure
    for tag in soup.find_all(['p', 'div', 'br', 'li','h1', 'h2', 'h3', 'h4' ]):
        tag.insert_after(soup.new_string('\n\n'))
    
    text = soup.get_text(separator=" ")
    
    # Clean up
    text = html.unescape(text)                    # Convert &amp; → &, &#39; to ', etc.
    text = re.sub(r'\n\s*\n+', '\n\n', text)      # Normalize blank lines
    text = re.sub(r'[ \t]+', ' ', text)           # Multiple spaces to single space
    text = re.sub(r'[ \t]+$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^[ \t]+', '', text, flags=re.MULTILINE)
    text = text.strip()
    
    return text

In [25]:
# Sanity check to onfim if clean funciton is working
test_ids = [0, 1, 500, 750, 1000, 1500, 2000]

for i in test_ids:
    topic = all_topics[i]
    print("\n" + "=" * 60)
    print("TITLE:", topic['title'])
    print("RAW (first 400 chars)")
    print(topic['summary_raw'][:400])
    print("CLEANED (first 400 chars)")
    print(clean_summary(topic['summary_raw'])[:400])


TITLE: A1C
RAW (first 400 chars)
<p>A1C is a blood test for <a href="https://medlineplus.gov/diabetestype2.html">type 2 diabetes</a> and <a href="https://medlineplus.gov/prediabetes.html">prediabetes</a>. It measures your average blood glucose, or <a href="https://medlineplus.gov/bloodglucose.html">blood sugar</a>, level over the past 3 months. Doctors may use the A1C alone or in combination with other diabetes tests to make a di
CLEANED (first 400 chars)
A1C is a blood test for type 2 diabetes and prediabetes . It measures your average blood glucose, or blood sugar , level over the past 3 months. Doctors may use the A1C alone or in combination with other diabetes tests to make a diagnosis. They also use the A1C to see how well you are managing your diabetes. This test is different from the blood sugar checks that people with diabetes do every day.

TITLE: Abdominal Pain
RAW (first 400 chars)
<p>Your abdomen extends from below your chest to your groin. Some people call it the stomach,

In [27]:
# full-dataset check: any leftover HTML-looking tags after cleaning?
leftover = 0
for topic in all_topics:
    if re.search(r'<[a-zA-Z/][^>]*>', clean_summary(topic['summary_raw'])):
        leftover += 1
print(f"\ntopics with leftover HTML-looking tags after cleaning: {leftover} / {len(all_topics)}")
 


topics with leftover HTML-looking tags after cleaning: 0 / 2033


In [28]:
# apply cleaing + metadata to all topics
for topic in all_topics:
    topic['summary_clean'] = clean_summary(topic['summary_raw'])
    # Better word count (handles accented characters better)
    topic['word_count'] = len(re.findall(r"\b[\w'áéíóúñÁÉÍÓÚÑ]+\b", topic['summary_clean']))

print(f"\nCleaning successfully applied to all {len(all_topics)} topics.")
print("Sample word counts:", [t['word_count'] for t in all_topics[:5]])


Cleaning successfully applied to all 2033 topics.
Sample word counts: [199, 136, 88, 86, 130]


## Decide Scope — ALL English topics (per supervisor)

Supervisor directive (meeting w/c 13 Jul): do NOT hand-pick a subset of
topic groups — we don't have the domain expertise to judge what's
"discharge-relevant", and it introduces bias. Keep every English topic.
English only. 

In [55]:
final_topics = english_topics
print(f"final corpus: {len(final_topics)} English topics ")


final corpus: 1016 English topics 


## Save cleaned scope dataset to structured JSON

In [56]:
output_records = []
for t in final_topics:
    output_records.append({
        'id': t['id'],
        'title': t['title'],
        'url': t['url'],
        'summary': t['summary_clean'],
        'word_count': t['word_count'],
        'groups': t['groups'],
        'also_called': t['also_called'],
        'sites': t['sites'],
    })

In [58]:
with open('topics_english_all.json', 'w', encoding='utf-8') as f:
    json.dump(output_records, f, indent=2, ensure_ascii=False)

print(f"saved {len(output_records)} topics to topics_english_all.json")

saved 1016 topics to topics_english_all.json


## Extract stratified gold subset for testing

Now that we cover every group, the gold subset (for manual retrieval checks and later evaluation) should sample ACROSS groups, not be hand-picked


In [61]:
GOLD_SIZE = 30
random.seed(42)   # reproducibility


In [62]:
by_group = defaultdict(list)
for t in output_records:
    key = t['groups'][0] if t['groups'] else 'Ungrouped'
    by_group[key].append(t)

In [63]:
groups_sorted = sorted(by_group, key=lambda g: -len(by_group[g]))

In [64]:
gold = []
for g in groups_sorted:                      # one from every group first
    if len(gold) < GOLD_SIZE:
        gold.append(random.choice(by_group[g]))

i = 0
while len(gold) < GOLD_SIZE:                  # top up from the biggest groups
    pool = [t for t in by_group[groups_sorted[i % len(groups_sorted)]] if t not in gold]
    if pool:
        gold.append(random.choice(pool))
    i += 1

In [65]:
with open('gold_subset.json', 'w', encoding='utf-8') as f:
    json.dump(gold, f, indent=2, ensure_ascii=False)

print(f"gold subset: {len(gold)} topics across "
      f"{len(set(t['groups'][0] if t['groups'] else 'Ungrouped' for t in gold))} groups")
for t in gold:
    print(f"  - {t['title']}  [{t['groups'][0] if t['groups'] else 'Ungrouped'}]")

gold subset: 30 topics across 30 groups
  - Tetanus  [Infections]
  - Blood Count Tests  [Blood, Heart and Circulation]
  - Attention Deficit Hyperactivity Disorder  [Children and Teenagers]
  - Infectious Arthritis  [Bones, Joints and Muscles]
  - Heartburn  [Digestive System]
  - Guillain-Barre Syndrome  [Brain and Nerves]
  - Carcinoid Tumors  [Cancers]
  - Vulvar Disorders  [Women]
  - COPD  [Older Adults]
  - Burns  [Skin, Hair and Nails]
  - Schizophrenia  [Mental Health and Behavior]
  - Body Weight  [Food and Nutrition]
  - Alpha-1 Antitrypsin Deficiency  [Lungs and Breathing]
  - Alcohol Use Disorder (AUD) Treatment  [Substance Use and Disorders]
  - Eye Infections  [Eyes and Vision]
  - Endocrine Diseases  [Endocrine System]
  - Secondhand Smoke  [Poisoning, Toxicology, Environmental Health]
  - Bladder Diseases  [Kidneys and Urinary System]
  - Veterans and Military Health  [Health System]
  - Food Allergy  [Immune System]
  - Mobility Aids  [Wellness and Lifestyle]
  - Marf